In [3]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [5]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("/Users/isishassan/Documents/Ironhack/Week7/Labs/Day1/lab-natural-language-processing/data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


### Let's divide the training and test set into two partitions

In [6]:
X = data['text']
y = data['label']

In [7]:
# Your code
from sklearn.model_selection import train_test_split

#split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Data Preprocessing

In [8]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [9]:
X.head(15)

0     DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...
1                                              Will do.
2     Nora--Cheryl has emailed dozens of memos about...
3     Dear Sir=2FMadam=2C I know that this proposal ...
4                                                   fyi
5     sure -- bottom line - you need a special secur...
6     Dear Sir,I am Engr. Ugo Nzego with the Enginee...
7     Abedin Huma <AbedinH@state.gov>Saturday Novemb...
8     There is an Oct 16th George Marshall event at ...
9     <P>1 25% for you as the account owner <BR>2 65...
10    STRONG><A href=3D"http://www.cnn.com/2003/WORL...
11    Dear Friend,My name is Edward Moore QC.Princip...
12    Compliment, How are you doing today, Hope you ...
13                                        Who wrote it?
14     accident. <BR>&nbsp;<BR>On further investigat...
Name: text, dtype: object

In [10]:
from bs4 import BeautifulSoup, Comment

def clean_html(html):
    soup = BeautifulSoup(html, "lxml")
    
    # 1. Remove script and style tags (inline JS/CSS blocks)
    for tag in soup(["script", "style"]):
        tag.decompose()
    
    # Remove inline JS event handlers and style attributes
    for tag in soup.find_all(True):
        attrs_to_remove = []
        for attr in tag.attrs:
            if attr.lower().startswith("on") or attr.lower() == "style":
                attrs_to_remove.append(attr)
        for attr in attrs_to_remove:
            del tag.attrs[attr]
    
    # 2. Remove HTML comments
    for comment in soup.find_all(string=lambda text: isinstance(text, Comment)):
        comment.extract()
    
    # 3. Remove remaining tags but keep text
    text = soup.get_text(separator=" ")
    
    return text.strip()

X_clean_html = X.apply(clean_html)

/var/folders/vl/4lkjvsxx3pv_m7ynmbqrfd780000gn/T/ipykernel_1168/572155851.py:4: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  soup = BeautifulSoup(html, "lxml")


- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [11]:
# Your code

#remove special characters and numbers
def remove_special_and_numbers(text):
    return ''.join(
        ch if ch.isalpha() or ch.isspace() else ' '
        for ch in text
    )

X_clean_html = X_clean_html.apply(remove_special_and_numbers)

In [12]:
import re

def more_cleaning(text):
    # 1️⃣ Remove prefixed b (byte string artifact)
    if text.startswith("b'") or text.startswith('b"'):
        text = text[2:-1]
    elif text.startswith("b "):
        text = text[2:]
    
    # 2️⃣ Remove single-character words at start
    words = text.split()
    while words and len(words[0]) == 1:
        words.pop(0)
    text = " ".join(words)
    
    # 3️⃣ Replace multiple spaces with single space
    text = re.sub(r"\s+", " ", text)
    
    # 4️⃣ Convert to lowercase
    text = text.lower()
    
    return text.strip()

X_clean = X_clean_html.apply(more_cleaning)

## Now let's work on removing stopwords
Remove the stopwords.

In [19]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.tokenize import word_tokenize

nltk.download('stopwords')  # run once

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/isishassan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [16]:
# Your code
X_clean = X_clean.apply(
    lambda doc: " ".join(
        word for word in doc.split()
        if word not in stop_words
    )
)

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [20]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/isishassan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/isishassan/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/isishassan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/isishassan/nltk_data...


True

In [21]:
def get_wordnet_pos(word):
    """Map POS tag to first character lemmatize() accepts"""
    tag = nltk.pos_tag([word])[0][1][0]
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV}
    
    return tag_dict.get(tag, wordnet.NOUN)

wordnet_lemma = WordNetLemmatizer()

def lemmatize_text(text):
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    
    lemmas = [
        wordnet_lemma.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged
    ]
    
    return " ".join(lemmas)

In [58]:
# quick sanity check
lemmatize_text("dogs are running and ate the cakes")
# → 'dog be run and eat the cake'

# now transform the whole series
X_lemm = X_clean.apply(lemmatize_text)       # or overwrite X_clean if you prefer
# X_clean = X_clean.apply(lemmatize_text)

In [33]:
X_lemm.head()

0    dear sir strictly private business proposal mi...
1                                                     
2    nora cheryl emailed dozen memo haiti weekend p...
3    dear sir fmadam c know proposal might surprise...
4                                                  fyi
Name: text, dtype: object

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [57]:
#get labels back in somehow

df_clean = pd.DataFrame({
    "text": X_lemm,
    "label": y
})

In [37]:
# Your code
spam_text = df_clean[df_clean["label"] == 1]["text"]
ham_text  = df_clean[df_clean["label"] == 0]["text"]

spam_words = " ".join(spam_text).split()
ham_words  = " ".join(ham_text).split()

In [43]:
from collections import Counter

spam_counter = Counter(spam_words)
ham_counter  = Counter(ham_words)

print(f"Spam: {spam_counter.most_common(10)}")
print(f"Ham: {ham_counter.most_common(10)}")

Spam: [('e', 2659), ('c', 1375), ('u', 989), ('money', 983), ('account', 896), ('bank', 795), ('fund', 777), ('f', 568), ('transaction', 555), ('business', 513)]
Ham: [('â', 239), ('u', 148), ('state', 134), ('pm', 126), ('would', 106), ('ã', 102), ('president', 99), ('time', 95), ('call', 94), ('mr', 91)]


## Extra features

In [45]:
df_clean.head()

,text,label
0,dear sir strictly private business proposal mi...,1
1,,0
2,nora cheryl emailed dozen memo haiti weekend p...,0
3,dear sir fmadam c know proposal might surprise...,1
4,fyi,0


In [ ]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

#data['money_mark'] = data['text'].str.contains(money_simbol_list)*1
data['suspicious_words'] = data['text'].str.contains(suspicious_words)*1
data['text_len'] = data['text'].apply(lambda x: len(x)) 

#X_test['money_mark'] = X_test['preprocessed_text'].str.contains(money_simbol_list)*1
#X_test['suspicious_words'] = X_test['preprocessed_text'].str.contains(suspicious_words)*1
#X_test['text_len'] = X_test['preprocessed_text'].apply(lambda x: len(x)) 

data.head()

#wouldn't it have been way smarter to first do this? then do the train test split?

,text,label,money_mark,suspicious_words,text_len
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1,1,0,2292
1,Will do.,0,0,0,8
2,Nora--Cheryl has emailed dozens of memos about...,0,0,0,197
3,Dear Sir=2FMadam=2C I know that this proposal ...,1,1,1,2199
4,fyi,0,0,0,3


In [ ]:
#adding the cleaned version to my og df
data['text_clean'] = X_lemm

#i will need to re-split my train / test

## How would work the Bag of Words with Count Vectorizer concept?

In [68]:
# Your code
from sklearn.feature_extraction.text import CountVectorizer

# Convert processed_docs to strings
docs_as_strings = X_lemm.tolist()

vectorizer = CountVectorizer()                              #initialise vectoriser
X = vectorizer.fit_transform(docs_as_strings) 

In [70]:
print("Feature Names:", vectorizer.get_feature_names_out())
print("Document-Term Matrix:\n", X.toarray())

Feature Names: ['aa' 'aaa' 'aabeiawaeaambiqaceqedeqh' ... 'zzz' 'zzzahbxntxe' 'zzzj']
Document-Term Matrix:
 [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [ ]:
# Your code

## And the Train a Classifier?

In [ ]:
# Your code

### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
# Your code